## I2 reader mode: Historical accepted CONSTRUCTED trees display replay

All newly computed values are **CONSTRUCTED method demonstrations**, never market observations or historical verification. Historical tables are separately labelled replay. French raw/extracted/processed bytes are not bundled. French route: A10-A limited-scope design accepted, execution blocked; A10-A.1 protocol technically accepted pending permissions and execution authorization. G01 rights/subject/use unknown; G05 A10-B not authorized.

Original explanatory prose below describes the course method. The reader-mode code and labels define which portions execute here. No P6 retraining or Ch39 historical row reconstruction.


# Project 6 v2：树模型比较、重要性、稳定性与消融（A8-B，构造数据）

冻结配置 `configs/p06_trees_v2.json` 继承 `p06_base_v1.json` 的数据、标签、折、内层验证、组合规则与成本账本，只新增单树、随机森林、梯度提升（HistGradientBoosting，内置早停关闭、轮数在内层日期折上选择）与 PCA+Ridge 辅助消融。冻结时已看过 A8-A v1 构造结果（配置内登记），不是真实市场的事前预注册。

**状态**：真实数据 blocked；以下全部为教学构造数据（`data_class = CONSTRUCTED`）。完整 Project 6 未 verified。

时间：signal_time $=D_k$ 收盘；trade_time $=D_{k+1}$ 收盘；label_end $=$ label_available_at $=D_{k+6}$。

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("p06_trees_v2_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
P06 = ROOT / "fixtures/projects/project06"
print("P6 trees: historical accepted CONSTRUCTED result display; no fitting/retraining; real route blocked")


## 1. 缩小版完整流程（实时运行，约 20 秒）

8 只资产、420 日、每个模型一组参数；只为展示每一步都在同一折、同一账本上发生，数字不用于任何结论。

In [ ]:
print("Original small model fitting cell SKIPPED in I2: no P6 retraining and no historical Ch39 repair")


## 2. 冻结配置的完整结果（读取 CLI 记录，不重跑）

IC 在各模型自身有 IC 的日期上平均，ΔIC 在与 Ridge 的共同日期上平均；Lasso 与 Elastic Net 部分日期预测为常数，`IC_dates` 少于 1002，其 $t$ 不报告（见 `v2_ic_date_bases.json`）。

来源：`research_report/v2_summary_base.json` 与 `v2_summary_regime.json`，由 `run_project06_v2.py` 在 `research_report/v2_trees.md` 所列运行中写出；基线 v1 模型与 A8-A c3 摘要逐值一致（`baseline_check`）。

In [ ]:
# IC dates: own IC dates differ for sparse models (constant predictions on some dates). t-values are shown only for
# series with IC on every mature-label date; for the others the frozen summary's t was computed on a compressed
# series of valid dates and is not a trading-day NW(4) statistic (rework R01).
bases = json.loads((P06 / "v2_ic_date_bases.json").read_text())["scenarios"]
rows = []
for scen in ("base", "regime"):
    s = json.loads((P06 / f"v2_summary_{scen}.json").read_text())
    assert s["data_class"] == "CONSTRUCTED" and s["baseline_check"]["ok"]
    for m, r in s["metrics"].items():
        if m in ("zero", "train_mean"):
            continue
        d, b = r.get("delta_ic_vs_ridge", {}), bases[scen][m]
        assert b["own_mean_matches_frozen"] and b.get("delta_matches_frozen", True)
        contiguous = b["frozen_t_is_trading_day_hac"]
        rows.append({"scenario": scen, "model": m, "R2_vs_mean_%": 100 * r["prediction"]["oos_r2_vs_train_mean"],
                     "IC": r["prediction"]["ic_mean"], "IC_dates": b["own_ic_dates"],
                     "IC_t": r["prediction"]["ic_t"] if contiguous else np.nan,
                     "dIC_vs_ridge": d.get("mean"), "paired_dates": b.get("paired_dates"),
                     "ridge_on_paired": b.get("ridge_paired_mean"), "dIC_t": d.get("t_nw") if contiguous else np.nan,
                     "net10_SR": r["portfolio"]["0.001"]["sharpe"], "net20_SR": r["portfolio"]["0.002"]["sharpe"],
                     "turnover": r["portfolio"]["0.001"]["turnover_ann"]})
full = pd.DataFrame(rows).set_index(["scenario", "model"])
display(full.round(4))
print("NaN t = not reported: the series has dates without a defined IC (see v2_ic_date_bases.json).")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for a, scen in zip(ax, ("base", "regime")):
    t = full.loc[scen]
    a.bar(t.index, t["net10_SR"], color=["#247F82" if m in ("tree", "rf", "gbm") else "#7F7F7F" for m in t.index])
    a.axhline(0, color="#444444", lw=0.6); a.set_title(f"{scen}: Sharpe after 10 bp (CONSTRUCTED)"); a.tick_params(axis="x", rotation=60)
plt.tight_layout(); fig.savefig(RUN_DIR / "historical_constructed_trees_sharpe.png",dpi=130); plt.show()

## 3. 消融与重要性稳定性

In [ ]:
s = json.loads((P06 / "v2_summary_base.json").read_text())
abl = pd.DataFrame({k: {"IC": v["prediction"]["ic_mean"], "dIC_vs_full": v["delta_ic_vs_full"]["mean"],
                        "net10_SR": v["portfolio"]["0.001"]["sharpe"]} for k, v in s["ablation"].items()}).T
display(abl.round(4))
print({m: round(v["consecutive_rank_corr_mean"], 3) for m, v in s["importance_stability"].items()})
print({m: {sd: round(r["portfolio"]["0.001"]["sharpe"], 3) for sd, r in v.items() if sd != "prediction_corr"} for m, v in s["seeds"].items()})